# LeetCode 1170 — Compare Strings by Frequency of the Smallest Character bằng Python

Link đề: https://leetcode.com/problems/compare-strings-by-frequency-of-the-smallest-character/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Binary Search, Sorting

Skeleton LeetCode:

```python
class Solution:
    def numSmallerByFrequency(self, queries: list[str], words: list[str]) -> list[int]:
        ...
```

Ghi chú: đáp án là **mảng số nguyên** cùng độ dài `queries`. Định nghĩa `f(s)` = số lần xuất hiện của ký tự **nhỏ nhất theo thứ tự từ điển** trong `s`. Với mỗi truy vấn `q`, đếm số từ `W` trong `words` có `f(q) < f(W)` (**nhỏ hơn hẳn**). Điểm cốt lõi: tính `f` của mọi từ **một lần**, rồi trả lời truy vấn bằng **tìm kiếm nhị phân** trên mảng đã sort hoặc **mảng đếm + tổng hậu tố** (vì `f ≤ 10`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hàm `f(s)` (tần suất ký tự nhỏ nhất) và phép so sánh **nghiêm ngặt** `f(q) < f(W)`.
2. Nắm công cụ: `min(s)`, `s.count(c)`, `sorted`, `bisect.bisect_right`, mảng đếm (counting) và **tổng hậu tố** (suffix sum).
3. Đi từ **tính lại mọi thứ cho mỗi cặp** → **tính `f(words)` trước rồi đếm tuyến tính** → **sort + bisect** → **đếm + tổng hậu tố `O(1)` mỗi truy vấn**.
4. Hiểu mẫu "**tiền xử lý một lần, trả lời nhiều truy vấn nhanh**" (offline/precompute).
5. Tránh lỗi: dùng `bisect_left` thay `bisect_right`, nhầm "ký tự nhỏ nhất" với "ký tự xuất hiện ít nhất", so sánh `<=` thay `<`.

## 2. Đề bài & ví dụ

Gọi `f(s)` là **tần suất** (số lần xuất hiện) của ký tự **nhỏ nhất theo thứ tự từ điển** trong chuỗi khác rỗng `s`. Ví dụ `s = "dcce"`: ký tự nhỏ nhất là `'c'`, xuất hiện 2 lần → `f(s) = 2`.

Cho mảng `words` và mảng truy vấn `queries`. Với mỗi `queries[i]`, đếm số từ `W` trong `words` sao cho `f(queries[i]) < f(W)`. Trả về mảng `answer` với `answer[i]` là kết quả truy vấn `i`.

**Ví dụ 1:** `queries = ["cbd"]`, `words = ["zaaaz"]` → `[1]`. `f("cbd") = 1` (`'b'` một lần), `f("zaaaz") = 3` → `1 < 3`.

**Ví dụ 2:** `queries = ["bbb","cc"]`, `words = ["a","aa","aaa","aaaa"]` → `[1,2]`. `f("bbb") = 3` chỉ nhỏ hơn `f("aaaa") = 4`; `f("cc") = 2` nhỏ hơn `3` và `4`.

**Ràng buộc:** `1 <= queries.length, words.length <= 2000`; `1 <= độ dài mỗi chuỗi <= 10`; chỉ chữ thường.

In [1]:
import random
import time
from bisect import bisect_left, bisect_right
from collections import Counter
from typing import List

examples = [
    (["cbd"], ["zaaaz"], [1]),
    (["bbb", "cc"], ["a", "aa", "aaa", "aaaa"], [1, 2]),
]
for q, w, want in examples:
    print(q, w, "->", want)

['cbd'] ['zaaaz'] -> [1]
['bbb', 'cc'] ['a', 'aa', 'aaa', 'aaaa'] -> [1, 2]


## 3. Kiến thức nền

### 3.1 `min` trên chuỗi và `str.count`

`min("dcce")` trả ký tự nhỏ nhất theo mã Unicode (với chữ thường chính là thứ tự từ điển) → `'c'`. `"dcce".count('c')` → `2`. Ghép lại được `f`.

In [2]:
def f(s):
    return s.count(min(s))

for s in ["dcce", "zaaaz", "cbd", "bbb", "a"]:
    print(s, "min =", min(s), "f =", f(s))
assert f("dcce") == 2 and f("zaaaz") == 3 and f("cbd") == 1

dcce min = c f = 2
zaaaz min = a f = 3
cbd min = b f = 1
bbb min = b f = 3
a min = a f = 1


### 3.2 Tính `f` bằng một lượt duyệt

Không dùng hàm có sẵn: duyệt từng ký tự, nếu nhỏ hơn ký tự nhỏ nhất hiện tại thì đặt lại bộ đếm về 1; nếu bằng thì tăng.

In [3]:
def f_manual(s):
    best, cnt = "{", 0           # '{' đứng ngay sau 'z' trong bảng mã
    for ch in s:
        if ch < best:
            best, cnt = ch, 1
        elif ch == best:
            cnt += 1
    return cnt

assert all(f_manual(s) == f(s) for s in ["dcce", "zaaaz", "cbd", "bbb", "a", "abcabc"])
print("f_manual khớp f")

f_manual khớp f


### 3.3 `sorted` và `bisect_right`

Trên mảng đã sort `arr`, `bisect_right(arr, x)` trả vị trí đầu tiên có phần tử **lớn hơn hẳn** `x`. Do đó số phần tử `> x` là `len(arr) − bisect_right(arr, x)`. (`bisect_left` trả vị trí đầu tiên `>= x`.)

In [4]:
arr = [1, 2, 2, 3, 4]
x = 2
print("bisect_left:", bisect_left(arr, x), "bisect_right:", bisect_right(arr, x))
print("số phần tử > 2:", len(arr) - bisect_right(arr, x))
assert len(arr) - bisect_right(arr, x) == 2

bisect_left: 1 bisect_right: 3
số phần tử > 2: 2


### 3.4 Mảng đếm và tổng hậu tố (suffix sum)

Khi giá trị nằm trong miền nhỏ (`f ∈ [1, 10]`), lập `cnt[v]` = số từ có `f = v`. Tổng hậu tố `suf[v] = cnt[v] + cnt[v+1] + …` cho số từ có `f ≥ v`. Số từ có `f > x` chính là `suf[x + 1]` — tra cứu `O(1)`.

In [5]:
vals = [1, 2, 3, 4]               # f của ["a","aa","aaa","aaaa"]
cnt = [0] * 12
for v in vals:
    cnt[v] += 1
suf = [0] * 12
for v in range(10, -1, -1):
    suf[v] = suf[v + 1] + cnt[v]
print("cnt:", cnt[:6], "suf:", suf[:6])
print("số từ có f > 3:", suf[4], "| f > 2:", suf[3])
assert suf[4] == 1 and suf[3] == 2

cnt: [0, 1, 1, 1, 1, 0] suf: [4, 4, 3, 2, 1, 0]
số từ có f > 3: 1 | f > 2: 2


### 3.5 List comprehension

Viết gọn biến đổi mảng: `[f(w) for w in words]`. Dùng xuyên suốt để tính `f` hàng loạt.

In [6]:
print([f(w) for w in ["a", "aa", "aaa", "aaaa"]])

[1, 2, 3, 4]


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2:

| Từ | ký tự nhỏ nhất | f |
|----|----------------|---|
| a | a | 1 |
| aa | a | 2 |
| aaa | a | 3 |
| aaaa | a | 4 |

Truy vấn `"bbb"`: `f = 3` → từ có `f > 3`: chỉ `"aaaa"` → `1`. Truy vấn `"cc"`: `f = 2` → `"aaa"`, `"aaaa"` → `2`.

Nhận xét: **chỉ giá trị `f` quan trọng**, bản thân chuỗi không còn ý nghĩa sau khi tính `f`. Bài toán thu về: cho mảng số `F_words`, với mỗi số `x` hỏi "có bao nhiêu phần tử `> x`?" — câu hỏi kinh điển giải bằng sort + nhị phân hoặc đếm.

In [7]:
F_words = [f(w) for w in examples[1][1]]
for q in examples[1][0]:
    x = f(q)
    print(q, "f =", x, "-> các f lớn hơn:", [v for v in F_words if v > x])

bbb f = 3 -> các f lớn hơn: [4]
cc f = 2 -> các f lớn hơn: [3, 4]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi truy vấn và mỗi từ, tính `f` của cả hai rồi so sánh — tính lại `f(W)` cho **mỗi** truy vấn.

In [8]:
def num_smaller_brute(queries, words):
    ans = []
    for q in queries:
        c = 0
        for w in words:
            if f(q) < f(w):
                c += 1
        ans.append(c)
    return ans

for q, w, want in examples:
    got = num_smaller_brute(q, w)
    print(got, want)
    assert got == want

[1] [1]
[1, 2] [1, 2]


**Độ phức tạp:** `O(Q · W · L)` với `L ≤ 10` → `2000 · 2000 · 10 · 2 = 8·10^7` thao tác — chậm trong Python (vài chục giây).

**Nút thắt:** `f(w)` được tính lại `Q` lần cho mỗi từ; `f(q)` tính lại `W` lần. Tính lại vô ích.

## 6. Bước đột phá

> **Tiền xử lý một lần:** `F = [f(w) for w in words]` (chỉ `W` lần tính). Sau đó mỗi truy vấn chỉ là câu hỏi đếm "bao nhiêu phần tử của `F` lớn hơn `x`".

Hai cách trả lời nhanh câu hỏi đếm:

- **Sort `F`** một lần; với `x`, đáp án `= W − bisect_right(F, x)` (`O(log W)`).
- **Đếm:** vì `1 ≤ f ≤ 10`, lập `suf[v]` = số phần tử `≥ v`; đáp án `= suf[x + 1]` (`O(1)`).

Quy tắc "nếu… thì…": nếu `F` đã sort và `F[k] > x` thì mọi `F[k'] (k' ≥ k)` cũng `> x` → tập phần tử `> x` là một **hậu tố** của mảng sort → chỉ cần tìm điểm bắt đầu hậu tố.

In [9]:
F = sorted(f(w) for w in examples[1][1])
for q in examples[1][0]:
    x = f(q)
    k = bisect_right(F, x)
    print(f"F={F} x={x} -> hậu tố bắt đầu tại {k}: {F[k:]} -> đếm {len(F) - k}")

F=[1, 2, 3, 4] x=3 -> hậu tố bắt đầu tại 3: [4] -> đếm 1
F=[1, 2, 3, 4] x=2 -> hậu tố bắt đầu tại 2: [3, 4] -> đếm 2


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tiền xử lý `f`, đếm tuyến tính

In [10]:
def num_smaller_precompute(queries, words):
    F = [f(w) for w in words]
    ans = []
    for q in queries:
        x = f(q)
        ans.append(sum(1 for v in F if v > x))
    return ans

for q, w, want in examples:
    assert num_smaller_precompute(q, w) == want
print("precompute OK")

precompute OK


**Độ phức tạp:** `O((Q + W) · L + Q · W)` = cỡ `4·10^6` — đã chạy được. Bộ nhớ `O(W)`.

### 7.2 Cách 3 — Sort + tìm kiếm nhị phân

In [11]:
def num_smaller_bisect(queries, words):
    F = sorted(f(w) for w in words)
    W = len(F)
    return [W - bisect_right(F, f(q)) for q in queries]

for q, w, want in examples:
    assert num_smaller_bisect(q, w) == want
print("bisect OK")

bisect OK


**Độ phức tạp:** `O(W log W)` sort + `O(Q log W)` truy vấn + `O((Q + W) · L)` tính `f`. Bộ nhớ `O(W)`.

### 7.3 Cách 4 — Mảng đếm + tổng hậu tố

`f` nằm trong `[1, 10]` (độ dài chuỗi ≤ 10). Dùng mảng kích thước 12 để có `suf[11] = 0` làm "lính canh" (sentinel) khi `x = 10`.

In [12]:
def num_smaller_count(queries, words):
    MAXF = 10
    cnt = [0] * (MAXF + 2)
    for w in words:
        cnt[f(w)] += 1
    suf = [0] * (MAXF + 2)
    for v in range(MAXF, 0, -1):
        suf[v] = suf[v + 1] + cnt[v]
    return [suf[f(q) + 1] for q in queries]

for q, w, want in examples:
    assert num_smaller_count(q, w) == want
print("count OK")

count OK


**Độ phức tạp:** `O((Q + W) · L + MAXF)` — tuyến tính theo tổng độ dài input. Bộ nhớ `O(MAXF)`.

### 7.4 Cách 5 — Hai con trỏ trên truy vấn đã sort (offline)

Sort cả `F_words` và **truy vấn kèm chỉ số gốc**. Duyệt truy vấn theo `f` tăng dần; con trỏ trên `F_words` chỉ tiến. Đây là kỹ thuật **xử lý truy vấn offline** — hữu ích khi không thể dùng mảng đếm (miền giá trị lớn) và muốn tránh `log`.

In [13]:
def num_smaller_two_pointers(queries, words):
    F = sorted(f(w) for w in words)
    order = sorted(range(len(queries)), key=lambda i: f(queries[i]))
    ans = [0] * len(queries)
    p = 0                                   # F[:p] là các phần tử <= x hiện tại
    for i in order:
        x = f(queries[i])
        while p < len(F) and F[p] <= x:
            p += 1
        ans[i] = len(F) - p
    return ans

for q, w, want in examples:
    assert num_smaller_two_pointers(q, w) == want
print("two pointers OK")

two pointers OK


**Độ phức tạp:** `O(W log W + Q log Q)` cho sort, phần quét `O(Q + W)`. Bộ nhớ `O(Q + W)`.

## 8. TRACE từng bước

Trace cách 4 với `queries = ["bbb", "cc", "zzzzzzzzzz", "abc"]`, `words = ["a", "aa", "aaa", "aaaa", "bcb"]`.

| Từ | f |
|----|---|
| a | 1 |
| aa | 2 |
| aaa | 3 |
| aaaa | 4 |
| bcb | 2 |

`cnt[1..5] = [1, 2, 1, 1, 0]` → `suf[1..5] = [5, 4, 2, 1, 0]`.

| Truy vấn | f(q) | suf[f+1] |
|----------|------|----------|
| bbb | 3 | suf[4] = 1 |
| cc | 2 | suf[3] = 2 |
| zzzzzzzzzz | 10 | suf[11] = 0 |
| abc | 1 | suf[2] = 4 |

In [14]:
def trace_count(queries, words):
    cnt = [0] * 12
    for w in words:
        cnt[f(w)] += 1
        print(f"word {w!r:8} f={f(w)} -> cnt[1..10]={cnt[1:11]}")
    suf = [0] * 12
    for v in range(10, 0, -1):
        suf[v] = suf[v + 1] + cnt[v]
    print("suf[1..11] =", suf[1:12])
    ans = []
    for q in queries:
        x = f(q)
        ans.append(suf[x + 1])
        print(f"query {q!r:12} f={x} -> suf[{x + 1}] = {suf[x + 1]}")
    return ans

res = trace_count(["bbb", "cc", "zzzzzzzzzz", "abc"], ["a", "aa", "aaa", "aaaa", "bcb"])
assert res == [1, 2, 0, 4]

word 'a'      f=1 -> cnt[1..10]=[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
word 'aa'     f=2 -> cnt[1..10]=[1, 1, 0, 0, 0, 0, 0, 0, 0, 0]
word 'aaa'    f=3 -> cnt[1..10]=[1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
word 'aaaa'   f=4 -> cnt[1..10]=[1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
word 'bcb'    f=2 -> cnt[1..10]=[1, 2, 1, 1, 0, 0, 0, 0, 0, 0]
suf[1..11] = [5, 4, 2, 1, 0, 0, 0, 0, 0, 0, 0]
query 'bbb'        f=3 -> suf[4] = 1
query 'cc'         f=2 -> suf[3] = 2
query 'zzzzzzzzzz' f=10 -> suf[11] = 0
query 'abc'        f=1 -> suf[2] = 4


Trace cách 3 (bisect) trên cùng dữ liệu:

In [15]:
def trace_bisect(queries, words):
    F = sorted(f(w) for w in words)
    print("F sorted =", F)
    for q in queries:
        x = f(q)
        k = bisect_right(F, x)
        print(f"query {q!r:12} x={x} bisect_right={k} -> {len(F) - k} (các phần tử {F[k:]})")

trace_bisect(["bbb", "cc", "zzzzzzzzzz", "abc"], ["a", "aa", "aaa", "aaaa", "bcb"])

F sorted = [1, 2, 2, 3, 4]
query 'bbb'        x=3 bisect_right=4 -> 1 (các phần tử [4])
query 'cc'         x=2 bisect_right=3 -> 2 (các phần tử [3, 4])
query 'zzzzzzzzzz' x=10 bisect_right=5 -> 0 (các phần tử [])
query 'abc'        x=1 bisect_right=1 -> 4 (các phần tử [2, 2, 3, 4])


## 9. Code nộp LeetCode

Chọn cách 4 (đếm + tổng hậu tố) — tuyến tính, ngắn.

In [16]:
from typing import List


class Solution:
    def numSmallerByFrequency(self, queries: List[str], words: List[str]) -> List[int]:
        def f(s: str) -> int:
            return s.count(min(s))

        cnt = [0] * 12                      # f nằm trong [1, 10]; suf[11] = 0 làm lính canh
        for w in words:
            cnt[f(w)] += 1
        suf = [0] * 12
        for v in range(10, 0, -1):
            suf[v] = suf[v + 1] + cnt[v]     # suf[v] = số từ có f >= v
        return [suf[f(q) + 1] for q in queries]


sol = Solution()
for q, w, want in examples:
    assert sol.numSmallerByFrequency(q, w) == want
print("Solution OK")

Solution OK


## 10. Vì sao đúng

1. **`f` đúng:** `min(s)` là ký tự nhỏ nhất theo thứ tự từ điển (chữ thường có mã ASCII tăng theo bảng chữ cái); `count` đếm đúng tần suất của nó.
2. **Miền giá trị:** `1 ≤ f(s) ≤ len(s) ≤ 10` (ký tự nhỏ nhất xuất hiện ít nhất một lần).
3. **Tổng hậu tố:** quy nạp từ `v = 10` xuống: `suf[11] = 0`; `suf[v] = cnt[v] + suf[v+1]` = số từ có `f = v` cộng số từ có `f ≥ v+1` = số từ có `f ≥ v`.
4. **Truy vấn:** số từ có `f(W) > x` = số từ có `f(W) ≥ x + 1` = `suf[x + 1]`. Với `x = 10`, `suf[11] = 0` — đúng, vì không từ nào có `f > 10`.

Với cách bisect: `F` sort tăng; `bisect_right(F, x)` = số phần tử `≤ x`; phần bù là số phần tử `> x`.

## 11. Time / Space Complexity

Gọi `Q = len(queries)`, `W = len(words)`, `L ≤ 10` là độ dài chuỗi tối đa.

| Cách | Time | Space |
|------|------|-------|
| 1. Brute | `O(Q · W · L)` | `O(Q)` (đầu ra) |
| 2. Precompute + đếm tuyến tính | `O((Q+W)L + Q·W)` | `O(W)` |
| 3. Sort + bisect | `O((Q+W)L + W log W + Q log W)` | `O(W)` |
| 4. Đếm + hậu tố | `O((Q+W)L)` | `O(L)` |
| 5. Hai con trỏ offline | `O((Q+W)L + W log W + Q log Q)` | `O(Q + W)` |

## 12. Bộ test

### 12.1 Ví dụ + edge case

In [17]:
impls = {
    "brute": num_smaller_brute,
    "precompute": num_smaller_precompute,
    "bisect": num_smaller_bisect,
    "count": num_smaller_count,
    "two_pointers": num_smaller_two_pointers,
    "Solution": Solution().numSmallerByFrequency,
}
edge = [
    (["a"], ["a"], [0]),                       # bằng nhau không tính
    (["a"], ["aa"], [1]),
    (["aaaaaaaaaa"], ["a", "bb", "zzzzzzzzzz"], [0]),   # f = 10 lớn nhất
    (["b"], ["ba", "ab", "aab"], [1]),         # ký tự nhỏ nhất là 'a', không phải ký tự đầu
    (["zyx", "xyzx"], ["xxyy", "yyyz"], [2, 1]),
    (["abcd"], ["dcba", "abcd"], [0]),
]
for name, fn in impls.items():
    for q, w, want in examples + edge:
        assert fn(q, w) == want, (name, q, w, fn(q, w))
print("all impls pass", len(examples + edge))

all impls pass 8


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Bảng chữ nhỏ để `f` đa dạng, độ dài 1–10.

In [18]:
rng = random.Random(1170)
def rand_word(rng):
    return "".join(rng.choice("abc") for _ in range(rng.randint(1, 10)))

for _ in range(400):
    q = [rand_word(rng) for _ in range(rng.randint(1, 8))]
    w = [rand_word(rng) for _ in range(rng.randint(1, 8))]
    want = num_smaller_brute(q, w)
    for name, fn in impls.items():
        assert fn(q, w) == want, (name, q, w)
print("random cross-check OK")

random cross-check OK


### 12.3 Test giới hạn

`Q = W = 2000`, chuỗi dài 10.

In [19]:
rng = random.Random(5)
Q = [ "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10)) for _ in range(2000)]
Wd = ["".join(rng.choice("abc") for _ in range(10)) for _ in range(2000)]
ref = None
for name in ["precompute", "bisect", "count", "two_pointers", "Solution"]:
    t0 = time.perf_counter()
    r = impls[name](Q, Wd)
    print(f"{name:12} sum={sum(r)} {time.perf_counter() - t0:.3f}s")
    ref = ref or r
    assert r == ref

precompute   sum=3487849 0.118s
bisect       sum=3487849 0.002s
count        sum=3487849 0.001s
two_pointers sum=3487849 0.002s


Solution     sum=3487849 0.002s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `bisect_left` (đếm cả phần tử bằng)

In [20]:
def wrong_bisect_left(queries, words):
    F = sorted(f(w) for w in words)
    return [len(F) - bisect_left(F, f(q)) for q in queries]

q, w = ["a"], ["a", "aa"]
print("sai:", wrong_bisect_left(q, w), "| đúng:", num_smaller_bisect(q, w))
assert wrong_bisect_left(q, w) != num_smaller_bisect(q, w)

sai: [2] | đúng: [1]


**Vì sao sai:** `bisect_left` đếm số phần tử `≥ x`, gồm cả `f(W) == f(q)`; đề yêu cầu **nhỏ hơn hẳn**. **Sửa:** `bisect_right`.

### Lỗi 2 — Nhầm "ký tự nhỏ nhất" với "ký tự xuất hiện ít nhất"

In [21]:
def wrong_f_least_frequent(s):
    return min(Counter(s).values())

s = "aab"
print("f sai:", wrong_f_least_frequent(s), "| f đúng:", f(s))
assert wrong_f_least_frequent(s) != f(s)

f sai: 1 | f đúng: 2


**Vì sao sai:** `"aab"` — ký tự nhỏ nhất (từ điển) là `'a'`, tần suất `2`; còn tần suất nhỏ nhất là `1` (của `'b'`). **Sửa:** `s.count(min(s))`.

### Lỗi 3 — Dùng ký tự **đầu tiên** của chuỗi

In [22]:
def wrong_f_first_char(s):
    return s.count(s[0])

s = "dcce"
print("f sai:", wrong_f_first_char(s), "| f đúng:", f(s))
assert wrong_f_first_char(s) != f(s)

f sai: 1 | f đúng: 2


**Vì sao sai:** chuỗi không được sắp; ký tự đầu không phải nhỏ nhất. **Sửa:** `min(s)`.

### Lỗi 4 — Mảng đếm thiếu lính canh (`IndexError` khi `f(q) = 10`)

In [23]:
def wrong_no_sentinel(queries, words):
    cnt = [0] * 11                         # chỉ số 0..10
    for w in words:
        cnt[f(w)] += 1
    suf = [0] * 11
    for v in range(10, 0, -1):
        suf[v] = (suf[v + 1] if v + 1 <= 10 else 0) + cnt[v]
    return [suf[f(q) + 1] for q in queries]

try:
    wrong_no_sentinel(["aaaaaaaaaa"], ["a"])
except IndexError as e:
    print("IndexError:", e)
print("đúng:", num_smaller_count(["aaaaaaaaaa"], ["a"]))

IndexError: list index out of range
đúng: [0]


**Vì sao sai:** truy cập `suf[11]` khi `f(q) = 10`. **Sửa:** cấp phát kích thước `12`, `suf[11] = 0`.

## 14. Mẹo & nhận dạng pattern

- **Nhiều truy vấn "đếm phần tử lớn hơn / nhỏ hơn x"** trên một tập cố định → sort + `bisect`, hoặc mảng đếm + tổng tiền tố/hậu tố khi miền giá trị nhỏ.
- **Tiền xử lý một lần**: mọi thứ không phụ thuộc truy vấn nên tính trước.
- `bisect_right` ↔ "nhỏ hơn hoặc bằng", `bisect_left` ↔ "nhỏ hơn hẳn" (đếm số phần tử bên trái).
- **Miền giá trị bị chặn nhỏ** (ở đây `≤ 10`) là gợi ý mạnh cho counting.
- Truy vấn có thể sort được (offline) → hai con trỏ.

## 15. Biến thể

1. Đếm `f(W) >= f(q)` hoặc `f(W) < f(q)` — chỉ đổi `bisect_left/right` hoặc tổng tiền tố.
2. **LC 2300 Successful Pairs of Spells and Potions** — sort một mảng, mỗi phần tử mảng kia tìm nhị phân ngưỡng.
3. **LC 1847 Closest Room** — truy vấn offline sort theo ngưỡng.
4. `f(s)` = tần suất ký tự **lớn nhất** — thay `min` bằng `max`.
5. Chuỗi dài tới `10^5` → miền `f` lớn, dùng bisect thay vì mảng đếm cố định (hoặc đếm với kích thước `max f + 2`).

In [24]:
# Biến thể 1: đếm số từ có f(W) <= f(q)
def num_le(queries, words):
    F = sorted(f(w) for w in words)
    return [bisect_right(F, f(q)) for q in queries]

print(num_le(["bbb", "cc"], ["a", "aa", "aaa", "aaaa"]))
assert num_le(["bbb", "cc"], ["a", "aa", "aaa", "aaaa"]) == [3, 2]

[3, 2]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Brute | Tính lại mọi `f` mỗi cặp | `O(QWL)` | `O(1)` thêm | Kiểm chứng |
| 2. Precompute | `F` một lần, đếm tuyến tính | `O(QW)` | `O(W)` | Input nhỏ |
| 3. Sort + bisect | Hậu tố của mảng sort | `O(W log W + Q log W)` | `O(W)` | Tổng quát, miền lớn |
| 4. Đếm + hậu tố | `suf[x+1]` | `O((Q+W)L)` | `O(L)` | Miền `f` nhỏ — **nộp bài** |
| 5. Offline 2 con trỏ | Sort truy vấn | `O(Q log Q + W log W)` | `O(Q+W)` | Khi cần duyệt truy vấn theo thứ tự |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `numSmallerByFrequency(["aabc", "z"], ["bb", "ccc", "a", "aaz"])` = ?

**Bài 2 (tính tay):** tính `cnt` và `suf` cho `words = ["xyx", "yy", "zzz", "w"]`.

**Bài 3 (điền TODO):** tự cài `bisect_right` bằng vòng lặp nhị phân (không dùng thư viện).

**Bài 4 (tự viết lại):** viết lại cách 4 nhưng dùng **tổng tiền tố** `pre[v]` = số từ có `f ≤ v`.

**Bài 5 (mini challenge):** với mỗi truy vấn, trả về số từ có `f(W)` **nằm trong** `[f(q) − 1, f(q) + 1]`.

In [25]:
# Bài 3 — TODO
def my_bisect_right(arr, x):
    # TODO: trả về vị trí đầu tiên có arr[i] > x
    raise NotImplementedError

# assert my_bisect_right([1, 2, 2, 3], 2) == 3
# assert my_bisect_right([1, 2, 2, 3], 0) == 0
# assert my_bisect_right([1, 2, 2, 3], 5) == 4

In [26]:
# Bài 5 — TODO
def num_in_band(queries, words):
    # TODO
    pass

# assert num_in_band(["cc"], ["a", "aa", "aaa", "aaaa"]) == [3]

## 18. Gợi ý 3 mức

- **Bài 1:** Mức 1: `f("aabc") = 2`, `f("z") = 1`. Mức 2: `f(words) = [2, 3, 1, 2]`. Mức 3: `> 2`: 1 từ; `> 1`: 3 từ → `[1, 3]`.
- **Bài 2:** Mức 1: `f = [2, 2, 3, 1]`. Mức 2: `cnt[1]=1, cnt[2]=2, cnt[3]=1`. Mức 3: `suf[1]=4, suf[2]=3, suf[3]=1, suf[4]=0`.
- **Bài 3:** Mức 1: `lo, hi = 0, len(arr)`. Mức 2: `mid`; nếu `arr[mid] <= x` thì `lo = mid + 1` else `hi = mid`. Mức 3: trả `lo`.
- **Bài 4:** Mức 1: `pre[0] = 0`. Mức 2: `pre[v] = pre[v-1] + cnt[v]`. Mức 3: đáp án `W − pre[x]`.
- **Bài 5:** Mức 1: dùng `pre`. Mức 2: số trong `[a, b]` = `pre[b] − pre[a−1]`. Mức 3: kẹp `a ≥ 1`, `b ≤ 10`.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert Solution().numSmallerByFrequency(["aabc", "z"], ["bb", "ccc", "a", "aaz"]) == [1, 3]

# Bài 2
cnt2 = [0] * 12
for w_ in ["xyx", "yy", "zzz", "w"]:
    cnt2[f(w_)] += 1
suf2 = [0] * 12
for v in range(10, 0, -1):
    suf2[v] = suf2[v + 1] + cnt2[v]
assert cnt2[1:4] == [1, 2, 1] and suf2[1:5] == [4, 3, 1, 0]

# Bài 3
def my_bisect_right_sol(arr, x):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] <= x:
            lo = mid + 1
        else:
            hi = mid
    return lo

rng = random.Random(3)
for _ in range(500):
    arr = sorted(rng.randint(0, 6) for _ in range(rng.randint(0, 8)))
    x = rng.randint(-1, 7)
    assert my_bisect_right_sol(arr, x) == bisect_right(arr, x)

# Bài 4
def num_smaller_prefix(queries, words):
    cnt = [0] * 11
    for w_ in words:
        cnt[f(w_)] += 1
    pre = [0] * 11
    for v in range(1, 11):
        pre[v] = pre[v - 1] + cnt[v]
    return [len(words) - pre[f(q)] for q in queries]

for q, w_, want in examples + edge:
    assert num_smaller_prefix(q, w_) == want
print("bài 1-4 OK")

bài 1-4 OK


In [28]:
# Bài 5
def num_in_band_sol(queries, words):
    cnt = [0] * 12
    for w_ in words:
        cnt[f(w_)] += 1
    pre = [0] * 12
    for v in range(1, 12):
        pre[v] = pre[v - 1] + cnt[v]
    res = []
    for q in queries:
        x = f(q)
        a, b = max(1, x - 1), min(10, x + 1)
        res.append(pre[b] - pre[a - 1])
    return res

assert num_in_band_sol(["cc"], ["a", "aa", "aaa", "aaaa"]) == [3]
rng = random.Random(55)
for _ in range(200):
    q = [rand_word(rng) for _ in range(5)]; w_ = [rand_word(rng) for _ in range(6)]
    assert num_in_band_sol(q, w_) == [sum(1 for ww in w_ if abs(f(ww) - f(qq)) <= 1) for qq in q]
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
f = lambda s: s.count(min(s))
cnt = [0] * 12
for w in words: cnt[f(w)] += 1
suf = [0] * 12
for v in range(10, 0, -1): suf[v] = suf[v + 1] + cnt[v]
return [suf[f(q) + 1] for q in queries]
# hoặc: F = sorted(map(f, words)); [len(F) - bisect_right(F, f(q)) for q in queries]
```

**Checklist:**

- [ ] `f` dùng ký tự **nhỏ nhất theo từ điển** (`min(s)`)?
- [ ] So sánh **nghiêm ngặt** `f(q) < f(W)` → `bisect_right` / `suf[x + 1]`?
- [ ] Mảng đếm đủ chỗ cho `x + 1 = 11`?
- [ ] Tính `f(words)` **một lần** (không trong vòng truy vấn)?
- [ ] Đầu ra giữ đúng thứ tự truy vấn ban đầu (nếu sort truy vấn, nhớ chỉ số)?